# 03 - Collaborative Filtering

**Purpose:** Implement and evaluate two collaborative filtering approaches:

- **Section A:** User-based Cosine Similarity CF (memory-based)
- **Section B:** Implicit ALS Matrix Factorization (model-based, via `implicit` library)

## Theory

**Cosine Similarity CF** finds users who have similar interaction patterns and
recommends items that similar users engaged with.

$$\text{sim}(u, v) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

**Implicit ALS** decomposes the user-item matrix `R ≈ U × V^T` using Alternating
Least Squares with confidence weighting on implicit feedback signals.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

In [ ]:
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
OUTPUT_DIR = '../output/slates'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('../output/evaluation', exist_ok=True)

print('Environment set up.')

## 1. Load Data & Build User-Item Matrix

In [ ]:
from scripts.load_data import build_user_item_matrix

def load_user_item_matrix(gold_dir: str) -> pd.DataFrame:
    """Load gold_user_item_matrix from Parquet or generate demo."""
    path = os.path.join(gold_dir, 'gold_user_item_matrix.parquet')
    if os.path.exists(path):
        print(f'[Parquet] Loading from {path}')
        return pd.read_parquet(path)
    print('[Demo] Generating synthetic user-item matrix...')
    rng = np.random.default_rng(42)
    n = 2000
    return pd.DataFrame({
        'user_id': rng.integers(1, 101, n),
        'node_id': rng.integers(100, 160, n),
        'implicit_affinity_score': rng.uniform(0.1, 5.0, n).round(3),
        'created_at': pd.date_range('2024-01-01', periods=n, freq='1h'),
    })

df_matrix_raw = load_user_item_matrix(GOLD_DIR)
print(f'Raw interaction data shape: {df_matrix_raw.shape}')
display(df_matrix_raw.head(5))

# Build pivot: rows=users, cols=items, values=affinity_score
pivot_df = build_user_item_matrix(df_matrix_raw)
print(f'\nUser-item pivot matrix shape: {pivot_df.shape}')
print(f'  Users (rows): {pivot_df.shape[0]:,}')
print(f'  Items (cols): {pivot_df.shape[1]:,}')
sparsity = (pivot_df == 0).sum().sum() / pivot_df.size
print(f'  Sparsity:     {sparsity:.1%}')

## Section A: Cosine Similarity Collaborative Filtering

In [ ]:
print('=== Section A: Cosine Similarity CF ===')

user_item_np = pivot_df.values  # shape: (n_users, n_items)
user_ids     = pivot_df.index.tolist()
item_ids     = pivot_df.columns.tolist()

# Compute user-user cosine similarity matrix
user_sim_matrix = cosine_similarity(user_item_np)  # (n_users, n_users)
print(f'User similarity matrix shape: {user_sim_matrix.shape}')

# Generate top-5 recommendations per user
K = 5
N_SIMILAR_USERS = 10  # top-N similar users to aggregate from

cf_predictions = {}  # user_id -> [node_id1, node_id2, ...]

for u_idx, user_id in enumerate(user_ids):
    sim_scores = user_sim_matrix[u_idx].copy()
    sim_scores[u_idx] = -1  # exclude self

    # Find top similar users
    top_sim_indices = np.argsort(sim_scores)[::-1][:N_SIMILAR_USERS]

    # Items the current user has already interacted with
    already_seen = set(np.where(user_item_np[u_idx] > 0)[0])

    # Aggregate scores from similar users (weighted by similarity)
    score_vector = np.zeros(len(item_ids))
    for sim_idx in top_sim_indices:
        weight = max(sim_scores[sim_idx], 0.0)
        score_vector += weight * user_item_np[sim_idx]

    # Zero out already-seen items
    for seen_idx in already_seen:
        score_vector[seen_idx] = 0.0

    # Top-K item indices
    top_item_indices = np.argsort(score_vector)[::-1][:K]
    cf_predictions[user_id] = [item_ids[i] for i in top_item_indices if score_vector[i] > 0]

n_with_recs = sum(1 for v in cf_predictions.values() if v)
print(f'Users with at least 1 CF recommendation: {n_with_recs:,} / {len(user_ids):,}')

# Show sample
sample_user = user_ids[0]
print(f'\nSample CF recommendations for user {sample_user}:')
print(cf_predictions[sample_user])

## Section B: Implicit ALS Matrix Factorization

Uses the `implicit` library for GPU-accelerated ALS on sparse implicit feedback data.
Skipped gracefully if `implicit` is not installed.

In [ ]:
als_predictions = {}

try:
    import implicit
    print(f'implicit version: {implicit.__version__}')

    # Build sparse user-item matrix (item x user for implicit library)
    user_item_sparse = sp.csr_matrix(user_item_np)  # (users, items)
    item_user_sparse = user_item_sparse.T.tocsr()    # (items, users) - required by implicit

    model = implicit.als.AlternatingLeastSquares(
        factors=64,
        regularization=0.01,
        iterations=20,
        calculate_training_loss=True,
        random_state=42,
    )
    model.fit(item_user_sparse)
    print('ALS model fitted.')

    for u_idx, user_id in enumerate(user_ids):
        rec_ids_raw, scores = model.recommend(
            u_idx,
            user_item_sparse[u_idx],
            N=K,
            filter_already_liked_items=True,
        )
        als_predictions[user_id] = [item_ids[i] for i in rec_ids_raw]

    print(f'ALS predictions generated for {len(als_predictions):,} users.')

except ImportError:
    print('[SKIP] `implicit` library not installed.')
    print('  Install with: pip install implicit')
    print('  Section B (ALS) will be skipped in evaluation.')

## Evaluation

In [ ]:
from scripts.metrics import evaluate_model, summarize_evaluation

# Ground truth: temporal split - last 20% of interactions per user
from scripts.load_data import train_test_split_temporal

if 'created_at' in df_matrix_raw.columns:
    train_df, test_df = train_test_split_temporal(df_matrix_raw, test_frac=0.2)
else:
    df_tmp = df_matrix_raw.copy()
    df_tmp['created_at'] = pd.date_range('2024-01-01', periods=len(df_tmp), freq='1h')
    train_df, test_df = train_test_split_temporal(df_tmp, test_frac=0.2)

ground_truth = (
    test_df.groupby('user_id')['node_id']
    .apply(list)
    .to_dict()
)
print(f'Test users: {len(ground_truth):,} | Total test interactions: {len(test_df):,}')

results = []

# Evaluate Cosine CF
cf_eval = evaluate_model(
    {u: preds for u, preds in cf_predictions.items() if u in ground_truth},
    ground_truth, k=5
)
cf_summary = summarize_evaluation(cf_eval)['mean']
results.append({
    'model': 'Cosine CF',
    **{col: cf_summary.get(col, float('nan')) for col in ['precision@5', 'recall@5', 'ndcg@5']}
})
print('\n[Cosine CF] Summary:')
display(summarize_evaluation(cf_eval))

# Evaluate ALS (if available)
if als_predictions:
    als_eval = evaluate_model(
        {u: preds for u, preds in als_predictions.items() if u in ground_truth},
        ground_truth, k=5
    )
    als_summary = summarize_evaluation(als_eval)['mean']
    results.append({
        'model': 'Implicit ALS',
        **{col: als_summary.get(col, float('nan')) for col in ['precision@5', 'recall@5', 'ndcg@5']}
    })
    print('\n[Implicit ALS] Summary:')
    display(summarize_evaluation(als_eval))

# Comparison table
comparison = pd.DataFrame(results)
print('\n=== CF Model Comparison ===')
display(comparison)

# Save CF slates
cf_rows = [{'user_id': uid, 'recommended_node_ids': str(nodes)}
           for uid, nodes in cf_predictions.items()]
pd.DataFrame(cf_rows).to_csv(os.path.join(OUTPUT_DIR, 'cf_cosine_slates.csv'), index=False)
print('\nCosine CF slates saved to output/slates/cf_cosine_slates.csv')